In [0]:
import json
import mlflow
import mlflow.deployments
import pyspark.sql.functions as sf

from databricks.vector_search.client import VectorSearchClient

from agent import build_agent, _load_config

mlflow.langchain.autolog()

In [0]:
df_bronze_products_info = spark.read.table(TABLE_BRONZE_PRODUCTS_INFO)
display(df_bronze_products_info.limit(10))

In [0]:
df_bronze_reviews = spark.read.table(TABLE_BRONZE_REVIEWS)
display(df_bronze_reviews.limit(10))

In [0]:
display(
    df_bronze_reviews
    .select("file_path")
    .distinct()
)

In [0]:
df_silver_products_recommendations = spark.read.table(TABLE_SILVER_PRODUCTS_RECOMMENDATIONS)
display(df_silver_products_recommendations.limit(10))

In [0]:
df_gold_products_recommendations = spark.read.table(TABLE_GOLD_PRODUCTS_RECOMMENDATIONS)
display(df_gold_products_recommendations)

In [0]:
vsc = VectorSearchClient()

deploy_client = mlflow.deployments.get_deploy_client("databricks")

question = "How buyers describe this product in their review?"
response = deploy_client.predict(
    endpoint=EMBEDDING_MODEL_ENDPOINT,
    inputs={"input": [question]}
)

embeddings = [
    e["embedding"]
    for e
    in  response.data
]

In [0]:
print("Embedding shape:", len(embeddings[0]))
print("Embeddings for question:", embeddings[0])

In [0]:
index = vsc.get_index(index_name=VECTOR_SEARCH_INDEX_NAME)
print(
    json.dumps(
        index.describe(), 
        indent=4
))

In [0]:
query_text = "Reviewer has mixed feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Reviewer has VERY negative feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Reviewer has VERY POSITIVE feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Reviewer has VERY POSITIVE feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    query_type="hybrid",
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Awkward, useless"
results = index.similarity_search(
    query_text=query_text,
    query_type="FULL_TEXT",
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
config = _load_config("agent_config_reviews.yaml")

agent = build_agent(
    llm_endpoint=config["llm_endpoint_name"],
    index_name=config["vs_index_name"],
    num_results=config["vs_num_results"],
    tool_name=config["tool_name"],
    tool_description=config["tool_description"],
    system_prompt=config["system_prompt"]
)

In [0]:
# Rebuild agent fresh to ensure we have the latest configuration
config = {"configurable": {"thread_id": "review_agent"}}
response = agent.invoke(
    input={
        "messages": [
            {
                "role": "user",
                "content": "What are top 3 MOST POSITIVE reviews?"
            },
            {
                "role": "user",
                "content": "What are top 3 MOST NEGATIVE reviews?"
            }
        ]
    },
    config=config
)
print(response["messages"][-1].content)

*Top 3 Most Positive Reviews**

| # | Review (excerpt) | Reason for classification |
|---|------------------|---------------------------|
| 1 | “I use this with my shiseido eye cream and eye serums and the next morning my eyes are so rested and plumped! **Love it**.” (ID 648370) | Explicitly positive language (“Love it”) and a clear benefit reported. |
| 2 | “Using the GloPro on my under‑eye skin has made a **big difference** for me.” (ID 648367) | Mention of a “big difference” indicates strong satisfaction. |
| 3 | “**Works great** on my face.” (ID 648366) | The phrase “works great” denotes a positive experience, even though the reviewer notes it is intense for sensitive skin. |

**Top 3 Most Negative Reviews**

| # | Review (excerpt) | Reason for classification |
|---|------------------|---------------------------|
| 1 | “I **hate** this attachment. … The width of the sides makes it nearly impossible to get it under your eye. … **It’s horrible**. I wish I had returned it.” (ID 648364) | Strong negative language (“hate”, “horrible”) and expression of regret. |
| 2 | “I have all the attachment heads … **except this one**. This one could’ve been great … **I hated** to do it but this sadly was returned.”